# 04_3 — Embedding Alignment Attack

The victim's database uses `all-mpnet-base-v2` embeddings — not GTR.
Vec2text can only invert GTR embeddings.

This notebook trains a linear projection from the mpnet embedding space to the GTR embedding space,
using paired encodings of the training corpus as supervision.
The projected target embeddings are then fed to vec2text.

**Prerequisite** : run `00_data_preparation.ipynb` and `00_target_data_preparation.ipynb` first.

Expected quality : lower than the native GTR attack in 04_2 — the projection is approximate.
The gap quantifies how much information is lost in alignment.

**Setup.** Configure the alignment and inversion parameters. `NUM_ALIGN_SAMPLES` controls how much of the training corpus is used to learn the projection — more data means better alignment.

In [ ]:
from gtr_runtime import load_gtr_encoder
import faiss
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import vec2text

TRAIN_DATA_FILE   = "data/sentences_train_text_db.parquet"
TRAIN_INDEX_FILE  = "data/sentences_train_vector_db.index"
TARGET_TEXT_FILE  = "data/sentences_target_text_db.parquet"
TARGET_INDEX_FILE = "data/sentences_target_vector_db.index"
GTR_MODEL_NAME    = "sentence-transformers/gtr-t5-base"

NUM_ALIGN_SAMPLES = 64_000   # full corpus — data volume is the main lever
ALIGN_HIDDEN      = 1024
ALIGN_EPOCHS      = 100
ALIGN_BATCH       = 1024     # larger batch for more stable gradients
ALIGN_LR          = 1e-4     # lower LR with full data

NUM_STEPS         = 20
PREFER_GPU        = True


def normalize_text(text: str) -> str:
    return " ".join(str(text).split())

**Build alignment pairs.** For each training sentence, compute both its mpnet embedding (from the index) and its GTR embedding (by encoding live). These paired vectors are the supervision signal for the projection.

In [ ]:
train_df = pd.read_parquet(TRAIN_DATA_FILE).sort_values("id").reset_index(drop=True)
mpnet_fi = faiss.read_index(TRAIN_INDEX_FILE)

sample_df = train_df.iloc[:NUM_ALIGN_SAMPLES]
X_mpnet   = np.zeros((len(sample_df), mpnet_fi.d), dtype=np.float32)
mpnet_fi.reconstruct_n(0, len(sample_df), X_mpnet)

if "gtr_encoder" not in vars():
    gtr_encoder, gtr_device = load_gtr_encoder(GTR_MODEL_NAME, prefer_gpu=PREFER_GPU)
X_gtr = gtr_encoder.encode(
    sample_df["text"].tolist(),
    batch_size=512,
    convert_to_numpy=True,
    normalize_embeddings=False,
    show_progress_bar=True,
)

print(f"mpnet: {X_mpnet.shape} | L2 norm sample = {np.linalg.norm(X_mpnet[0]):.4f}")
print(f"GTR: {X_gtr.shape} | L2 norm sample = {np.linalg.norm(X_gtr[0]):.4f}")

**Train the aligner.** Train a residual projection network (`mpnet → GTR`) using cosine similarity as the loss. Watch the validation cosine — it measures how faithfully the projection preserves semantic meaning across the two spaces.

In [ ]:
ALIGN_DEVICE = (
    torch.device("mps")  if torch.backends.mps.is_available() and PREFER_GPU else
    torch.device("cuda") if torch.cuda.is_available()         and PREFER_GPU else
    torch.device("cpu")
)
print(f"aligner device : {ALIGN_DEVICE}")

X_tr, X_val, Y_tr, Y_val = train_test_split(X_mpnet, X_gtr, test_size=0.2, random_state=42)

loader_tr = DataLoader(
    TensorDataset(torch.from_numpy(X_tr).float(), torch.from_numpy(Y_tr).float()),
    batch_size=ALIGN_BATCH,
    shuffle=True,
)
X_val_t = torch.from_numpy(X_val).float().to(ALIGN_DEVICE)
Y_val_t = torch.from_numpy(Y_val).float().to(ALIGN_DEVICE)

IN_DIM  = X_mpnet.shape[1]
OUT_DIM = X_gtr.shape[1]


class ResidualAligner(nn.Module):
    """Linear projection + small residual MLP from source embedding space to GTR space."""

    def __init__(self, in_dim: int, out_dim: int, hidden: int):
        super().__init__()
        self.linear   = nn.Linear(in_dim, out_dim)
        self.residual = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x) + self.residual(x)

    def predict(self, X: np.ndarray) -> np.ndarray:
        self.eval()
        with torch.no_grad():
            return self(torch.from_numpy(X).float().to(ALIGN_DEVICE)).cpu().numpy()


aligner   = ResidualAligner(IN_DIM, OUT_DIM, ALIGN_HIDDEN).to(ALIGN_DEVICE)
optimizer = torch.optim.Adam(aligner.parameters(), lr=ALIGN_LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ALIGN_EPOCHS)

for epoch in range(1, ALIGN_EPOCHS + 1):
    aligner.train()
    for xb, yb in loader_tr:
        xb, yb = xb.to(ALIGN_DEVICE), yb.to(ALIGN_DEVICE)
        loss   = (1.0 - nn.functional.cosine_similarity(aligner(xb), yb)).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    scheduler.step()

    if epoch % 10 == 0 or epoch == 1:
        aligner.eval()
        with torch.no_grad():
            val_cos = nn.functional.cosine_similarity(aligner(X_val_t), Y_val_t).mean().item()
        print(f"epoch {epoch}/{ALIGN_EPOCHS} | val cosine: {val_cos:.4f}")

aligner.eval()
with torch.no_grad():
    align_cosines = nn.functional.cosine_similarity(aligner(X_val_t), Y_val_t).cpu().numpy()

print(f"\nAlignment quality on {len(X_val)} held-out pairs :")
print(f"mean cosine: {align_cosines.mean():.4f}")
print(f"min cosine: {align_cosines.min():.4f}")

**Project the targets.** Load the target mpnet embeddings and map them to GTR space using the trained aligner. Then load the vec2text corrector.

In [ ]:
target_df = pd.read_parquet(TARGET_TEXT_FILE).sort_values("id").reset_index(drop=True)
target_fi = faiss.read_index(TARGET_INDEX_FILE)

target_mpnet      = np.zeros((target_fi.ntotal, target_fi.d), dtype=np.float32)
target_fi.reconstruct_n(0, target_fi.ntotal, target_mpnet)
target_gtr_mapped = aligner.predict(target_mpnet).astype(np.float32)

corrector        = vec2text.load_pretrained_corrector("gtr-base")
corrector_device = next(corrector.model.parameters()).device
print(f"{len(target_df)} targets | corrector on {corrector_device}")

**Attack.** Feed the projected embeddings to vec2text. Since the projection is approximate, expect lower cosine similarity than the native GTR attack in notebook 04_1. The gap quantifies information lost in alignment.

In [ ]:
results = []

for i, row in tqdm(enumerate(target_df.itertuples(index=False)), total=len(target_df)):
    mapped_emb    = torch.tensor(target_gtr_mapped[i], dtype=torch.float32).unsqueeze(0).to(corrector_device)
    reconstructed = vec2text.invert_embeddings(
        embeddings=mapped_emb,
        corrector=corrector,
        num_steps=NUM_STEPS,
    )[0]

    reconstructed_emb = gtr_encoder.encode([reconstructed], convert_to_tensor=True, normalize_embeddings=False)
    cosine = float(torch.nn.functional.cosine_similarity(
        mapped_emb.to(reconstructed_emb.device), reconstructed_emb
    ))
    results.append({
        "target_id":   row.target_id,
        "ground_truth": row.text,
        "reconstructed": reconstructed,
        "cosine":       cosine,
        "exact_match":  normalize_text(row.text) == normalize_text(reconstructed),
    })
    print(f"[{len(results):>2}/{len(target_df)}] {row.target_id:<12} cosine={cosine:.3f} | {normalize_text(reconstructed)!r}")

**Results.** Compare mean cosine and exact-match rate against notebook 04_1. What does the alignment cosine tell you about the upper bound on attack quality?

In [ ]:
exact_matches  = sum(r["exact_match"] for r in results)
attack_cosines = [r["cosine"] for r in results]

print(f"Alignment cosine (mpnet → GTR) : {align_cosines.mean():.3f}")
print(f"Attack mean cosine: {sum(attack_cosines)/len(attack_cosines):.3f} | Exact matches: {exact_matches}/{len(results)}\n")

for r in results:
    print("------")
    print(r["target_id"])
    print(f"original: {r['ground_truth']}")
    print(f"reconstructed: {normalize_text(r['reconstructed'])}")
    print(f"cosine: {r['cosine']:.3f}")
print("------")